In [3]:

import re

# Check whether a term is a variable
def is_variable(x):
    return x.islower()


# Parse a predicate
def parse_atom(atom):
    atom = atom.strip()
    match = re.fullmatch(r'(\w+)\((.*?)\)', atom)

    if not match:
        raise ValueError("Invalid predicate: " + atom)

    name = match.group(1)
    args = [x.strip() for x in match.group(2).split(",")]

    return name, args


# Unification
def unify(pattern, fact, substitution):
    p_name, p_args = parse_atom(pattern)
    f_name, f_args = parse_atom(fact)

    if p_name != f_name or len(p_args) != len(f_args):
        return None

    new_sub = substitution.copy()

    for p, f in zip(p_args, f_args):

        if is_variable(p):
            if p in new_sub:
                if new_sub[p] != f:
                    return None
            else:
                new_sub[p] = f

        elif p != f:
            return None

    return new_sub


# Substitute variables
def substitute(atom, sub):
    name, args = parse_atom(atom)

    new_args = [sub.get(x, x) for x in args]

    return name + "(" + ",".join(new_args) + ")"


# Forward chaining algorithm
def forward_chaining(facts, rules, query):

    known = set(facts)

    print("\nInitial Facts:")
    for f in sorted(known):
        print(f)

    iteration = 1

    while True:

        new_facts = set()

        for premises, conclusion in rules:

            substitutions = [{}]

            # Match premises with known facts
            for premise in premises:

                updated = []

                for sub in substitutions:
                    for fact in known:

                        result = unify(premise, fact, sub)

                        if result is not None:
                            updated.append(result)

                substitutions = updated

                if not substitutions:
                    break

            # Generate new conclusions
            for sub in substitutions:

                inferred = substitute(conclusion, sub)

                if inferred not in known:
                    new_facts.add(inferred)

        # Stop if no new facts
        if not new_facts:
            break

        print("\nIteration", iteration)

        for f in sorted(new_facts):
            print("New Fact:", f)

        known.update(new_facts)

        iteration += 1

    # Display final knowledge base
    print("\nFinal Knowledge Base:")

    for f in sorted(known):
        print(f)

    # Check query
    print("\nQuery:", query)

    if query in known:
        print("RESULT: QUERY PROVED (TRUE)")
    else:
        print("RESULT: QUERY NOT PROVED (FALSE)")


# ---------------- MAIN PROGRAM ----------------

print("FIRST ORDER LOGIC - FORWARD CHAINING")

# Initial facts
facts = [
    "Enemy(A,America)",
    "Owns(A,T1)",
    "Missile(T1)",
    "American(Robert)"
]

# Rules
rules = [
    (
        ["American(x)", "Weapon(y)", "Sells(x,y,z)", "Hostile(z)"],
        "Criminal(x)"
    ),
    (
        ["Missile(x)", "Owns(A,x)"],
        "Sells(Robert,x,A)"
    ),
    (
        ["Missile(x)"],
        "Weapon(x)"
    ),
    (
        ["Enemy(x,America)"],
        "Hostile(x)"
    )
]

# Query
query = "Criminal(Robert)"

# Execute forward chaining
forward_chaining(facts, rules, query)


FIRST ORDER LOGIC - FORWARD CHAINING

Initial Facts:
American(Robert)
Enemy(A,America)
Missile(T1)
Owns(A,T1)

Iteration 1
New Fact: Hostile(A)
New Fact: Sells(Robert,T1,A)
New Fact: Weapon(T1)

Iteration 2
New Fact: Criminal(Robert)

Final Knowledge Base:
American(Robert)
Criminal(Robert)
Enemy(A,America)
Hostile(A)
Missile(T1)
Owns(A,T1)
Sells(Robert,T1,A)
Weapon(T1)

Query: Criminal(Robert)
RESULT: QUERY PROVED (TRUE)
